In [1]:
#SETUP
import ConnectionConfig as cc
cc.setupEnvironment()

spark = cc.startLocalCluster("Analyzing_rides")
spark.getActiveSession()

In [2]:
#EXTRACT
dim_customer = spark.read.format("delta").load("spark-warehouse/dimCustomer")
dim_locks = spark.read.format("delta").load("spark-warehouse/dimLocks")
fact_rides = spark.read.format("delta").load("spark-warehouse/factrides")

AnalysisException: [PATH_NOT_FOUND] Path does not exist: spark-warehouse/factrides.

In [4]:
#TRANSFORM
#TEMP VIEW MAKEN  
dim_customer.createOrReplaceTempView("dimCustomer")
dim_locks.createOrReplaceTempView("dimLocks")
fact_rides.createOrReplaceTempView("factRides")

## Wat is de invloed van de woonplaats van de gebruikers op het gebruik van de vehicles?

In [18]:
influence_residence = spark.sql("""
    SELECT 
        c.city AS user_residence,
        COUNT(r.rideId) AS number_of_rides
    FROM 
        factRides AS r
    JOIN 
        dimCustomer AS c ON r.userId = c.userId
    GROUP BY 
        c.city
    ORDER BY 
        number_of_rides DESC
""")

influence_residence.show()

+--------------------+---------------+
|      user_residence|number_of_rides|
+--------------------+---------------+
|           Antwerpen|         552434|
|             Schoten|         136907|
|Beveren-Waas/Haas...|         134273|
| Wilrijk (Antwerpen)|         120687|
|  Deurne (Antwerpen)|         113363|
|          Brasschaat|          99782|
|    Kapellen (Antw.)|          97693|
| Berchem (Antwerpen)|          95935|
|Broechem/Emblem/O...|          94702|
|s Gravenwezel/Sch...|          92609|
| Merksem (Antwerpen)|          91646|
| Hoboken (Antwerpen)|          85166|
|    Kontich/Waarloos|          82344|
|Borgerhout (Antwe...|          80243|
|              Edegem|          78672|
|Bazel/Kruibeke/Ru...|          68176|
|Antwerpen/Berendr...|          68034|
| Reet/Rumst/Terhagen|          66446|
|            Boechout|          65628|
|             Mortsel|          63349|
+--------------------+---------------+


## We willen voorspellen welke sloten preventief onderhoud nodig hebben. Bekijk hoe vaak slotnummers relatief gezien gebruikt worden.


In [25]:
locks_maintenance = spark.sql("""
    SELECT 
        l.lockId,
        COUNT(r.rideId) AS number_of_rides
    FROM 
        factRides AS r
    JOIN 
        dimLocks AS l ON r.startLockId = l.lockId
    GROUP BY 
        l.lockId
    ORDER BY 
        number_of_rides DESC
""")
locks_maintenance.show()

+------+---------------+
|lockId|number_of_rides|
+------+---------------+
|  1548|           9235|
|  3258|           9176|
|  3024|           9169|
|   270|           9165|
|  2682|           9113|
|  2772|           9084|
|  1386|           9075|
|  2538|           9073|
|   702|           9068|
|  2718|           9068|
|   738|           9066|
|  1566|           9065|
|   809|           9044|
|  4230|           9043|
|  2969|           9041|
|  3132|           9040|
|  2898|           9037|
|   774|           9032|
|   665|           9031|
|  2790|           9022|
+------+---------------+


## Als een klant zijn abonnement stopzet, willen we kunnen voorspellen op welke stations dit het meeste effect zal hebben.

In [8]:
subscription_effect = spark.sql("""
    SELECT
        c.userId,
        l.lockId AS station,
        COUNT(r.rideId) AS number_of_rides
    FROM
        factRides AS r
    JOIN
        dimCustomer AS c ON r.userId = c.userId
    JOIN
        dimLocks AS l ON r.startLockId = l.lockId OR r.endLockId = l.lockId
    WHERE
        c.subscriptionid IS NOT NULL
    GROUP BY
        c.userId, l.lockId
    ORDER BY
        number_of_rides DESC
""")

subscription_effect.show()

+------+-------+---------------+
|userId|station|number_of_rides|
+------+-------+---------------+
| 10734|   1818|              9|
| 17038|   7505|              8|
| 17219|   1530|              8|
| 18723|   3617|              8|
| 51514|   3618|              8|
| 42927|   7505|              8|
| 53148|   3491|              8|
| 27610|   6497|              8|
| 38213|   3599|              8|
| 19260|   4158|              8|
| 53883|   4193|              8|
| 25217|   3493|              8|
| 31870|   3564|              8|
| 22084|   6498|              8|
| 47004|   3618|              8|
| 33889|   1476|              8|
| 55292|   2754|              8|
| 43788|    738|              8|
| 52381|   6498|              8|
| 42828|   3114|              8|
+------+-------+---------------+


## Wat is de gemiddelde afstand die wordt afgelegd per rit voor elke gebruiker?

In [5]:
average_distance_per_user = spark.sql("""
    SELECT
        c.userId,
        AVG(r.distance) AS average_distance
    FROM
        factRides AS r
    JOIN
        dimCustomer AS c ON r.userId = c.userId
    GROUP BY
        c.userId
    ORDER BY
        average_distance DESC
""")
average_distance_per_user.show()

+------+------------------+
|userId|  average_distance|
+------+------------------+
| 43173|12.241728782653809|
|   698|10.945701599121094|
| 37548|10.509639739990234|
|  3536|10.004549980163574|
| 57155| 9.705032348632812|
| 45532| 9.137563705444336|
|   916| 8.861483573913574|
| 44451| 8.755638122558594|
| 49150| 8.612909317016602|
|  6346| 8.457173347473145|
|  2269|  8.45273208618164|
| 56482| 8.330711364746094|
| 55995| 8.296283721923828|
|  1485|  8.22579288482666|
| 13277| 8.180845260620117|
| 10965|  8.14972972869873|
| 12105| 8.148478507995605|
| 31979| 8.124216079711914|
| 25735| 8.105899810791016|
| 14149| 8.087172508239746|
+------+------------------+


## Wat is het meest gebruikte start- en eindstation?

In [6]:
most_used_stations = spark.sql("""
    SELECT
        l.lockId AS station,
        COUNT(r.rideId) AS number_of_rides
    FROM
        factRides AS r
    JOIN
        dimLocks AS l ON r.startLockId = l.lockId OR r.endLockId = l.lockId
    GROUP BY
        l.lockId
    ORDER BY
        number_of_rides DESC
""")
most_used_stations.show()

+-------+---------------+
|station|number_of_rides|
+-------+---------------+
|   1548|          17726|
|    270|          17660|
|   2682|          17597|
|   3258|          17586|
|   4230|          17541|
|    702|          17538|
|   3024|          17536|
|   2772|          17533|
|    738|          17491|
|    774|          17450|
|   2538|          17441|
|   2718|          17434|
|   2825|          17394|
|    558|          17393|
|   1386|          17386|
|   6569|          17383|
|   2754|          17379|
|   2988|          17376|
|   2790|          17366|
|    810|          17356|
+-------+---------------+


## Wat is de gemiddelde duur van een rit per gebruiker?

In [7]:
average_duration_per_user = spark.sql("""
    SELECT
        c.userId,
        AVG(r.endtime - r.starttime) AS average_rides_duration
    FROM
        factRides AS r
    JOIN
        dimCustomer AS c ON r.userId = c.userId
    GROUP BY
        c.userId
    ORDER BY
        average_duration DESC
""")
average_duration_per_user.show()

+------+--------------------+
|userId|    average_duration|
+------+--------------------+
| 49150|INTERVAL '0 00:51...|
| 43173|INTERVAL '0 00:47...|
|   698|INTERVAL '0 00:45...|
|  2269|INTERVAL '0 00:43...|
| 30412|INTERVAL '0 00:39...|
| 44452|INTERVAL '0 00:39...|
| 55642|INTERVAL '0 00:39...|
|  3536|INTERVAL '0 00:39...|
| 14329|INTERVAL '0 00:39...|
| 29542|INTERVAL '0 00:39...|
| 45532|INTERVAL '0 00:39...|
| 54177|INTERVAL '0 00:37...|
| 44710|INTERVAL '0 00:37...|
|  2767|INTERVAL '0 00:37...|
| 57155|INTERVAL '0 00:37...|
| 13277|INTERVAL '0 00:37...|
| 37966|INTERVAL '0 00:37...|
|  7508|INTERVAL '0 00:37...|
| 56480|INTERVAL '0 00:37...|
| 26693|INTERVAL '0 00:36...|
+------+--------------------+
